# Efecto por año

## Preparación

In [1]:
from nbsetup import *
import statsmodels.formula.api as smf
from pipeline import Datos, muestra_comun, Panel, NIVELES_TESIS
from pipeline.estimacion import estrellas

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

datos = Datos()
emparejamientos, comunes = muestra_comun(datos)

ANIOS = {"año 1": (0, 11), "año 2": (12, 23), "año 3": (24, 35)}
PERIODOS = ["antes"] + list(ANIOS)


In [2]:
def panel_anios(emparejamiento, niveles=None):
    p = Panel(emparejamiento, niveles=niveles)
    tabla = p.tabla.copy()
    for k, (desde, hasta) in enumerate(ANIOS.values(), start=1):
        tabla[f"a{k}"] = tabla.t.between(desde, hasta).astype(int)
        tabla[f"trat_a{k}"] = tabla.tratado * tabla[f"a{k}"]
        for c in p.covariables:
            tabla[f"{c}_a{k}"] = tabla[f"z_{c}"] * tabla[f"a{k}"]
    tabla["periodo"] = np.select(
        [tabla.t < 0, tabla.a1 == 1, tabla.a2 == 1, tabla.a3 == 1], PERIODOS, default="")
    return tabla, p.covariables


def formulas(covariables):
    tratamiento = "incidentes ~ trat_a1 + trat_a2 + trat_a3"
    return {
        "sin covariables": tratamiento + " + C(uid) + C(t)",
        "covariables × año": (tratamiento + " + "
                              + " + ".join(f"{c}_a{k}" for k in (1, 2, 3) for c in covariables)
                              + " + C(uid) + C(t)"),
    }


filas, medias = [], []
for radio, e in emparejamientos.items():
    for nivel, niveles in NIVELES_TESIS.items():
        tabla, covariables = panel_anios(e, niveles)
        base = tabla[(tabla.tratado == 0) & (tabla.t < 0)].incidentes.mean()

        m = tabla.groupby(["tratado", "periodo"]).incidentes.mean().unstack()[PERIODOS]
        for tratado, grupo in [(1, "tratadas"), (0, "controles")]:
            medias.append({"radio": radio, "nivel": nivel, "grupo": grupo} | m.loc[tratado].to_dict())

        for especificacion, formula in formulas(covariables).items():
            r = smf.ols(formula, data=tabla).fit(
                cov_type="cluster", cov_kwds={"groups": pd.factorize(tabla.cluster)[0]})
            igualdad = r.wald_test("trat_a1 = trat_a2, trat_a2 = trat_a3", scalar=True)
            fila = {"radio": radio, "nivel": nivel, "especificacion": especificacion,
                    "N": int(r.nobs), "R2": r.rsquared, "clusters": tabla.cluster.nunique(),
                    "media control, antes": base, "p igualdad entre años": float(igualdad.pvalue)}
            for k in (1, 2, 3):
                fila |= {f"coef año {k}": r.params[f"trat_a{k}"], f"EE año {k}": r.bse[f"trat_a{k}"],
                         f"sig año {k}": estrellas(r.pvalues[f"trat_a{k}"]),
                         f"% año {k}": 100 * r.params[f"trat_a{k}"] / base}
            filas.append(fila)

por_anio = pd.DataFrame(filas)
medias = pd.DataFrame(medias)
print(f"{len(por_anio)} estimaciones")


40 estimaciones


## Verificación

In [3]:
# sin covariables, el promedio de los tres años tiene que ser el DiD con un solo post
dif = []
for radio, e in emparejamientos.items():
    for nivel, niveles in NIVELES_TESIS.items():
        p = Panel(e, niveles=niveles)
        unico = p.estimar(p.especificaciones["(2) + EF unidad y tiempo"])["coef"]
        tres = por_anio.query("radio == @radio and nivel == @nivel "
                              "and especificacion == 'sin covariables'").iloc[0]
        dif.append(abs(np.mean([tres[f"coef año {k}"] for k in (1, 2, 3)]) - unico))
print(f"diferencia máxima en las 20 combinaciones: {max(dif):.2e}")
assert max(dif) < 1e-8


diferencia máxima en las 20 combinaciones: 1.78e-14


## Medias por grupo y periodo

### general

In [4]:
medias.query("nivel == 'general'").set_index(["radio", "grupo"])[PERIODOS].round(3)

antes  año 1  año 2  año 3
radio grupo                                
100   tratadas   2.286  2.176  1.541  2.096
      controles  2.223  2.065  1.459  1.815
150   tratadas   2.984  2.812  2.004  2.897
      controles  2.949  2.988  2.082  2.728
200   tratadas   3.893  3.668  2.611  3.868
      controles  3.866  3.873  2.643  3.797
250   tratadas   5.073  4.848  3.429  5.017
      controles  5.027  4.978  3.504  5.062
300   tratadas   6.222  5.920  4.193  6.151
      controles  6.019  5.998  4.147  5.954

### MIN

In [5]:
medias.query("nivel == 'MIN'").set_index(["radio", "grupo"])[PERIODOS].round(3)

antes  año 1  año 2  año 3
radio grupo                                
100   tratadas   1.324  1.157  0.653  0.893
      controles  1.319  1.142  0.689  0.799
150   tratadas   1.737  1.497  0.862  1.236
      controles  1.710  1.628  0.960  1.206
200   tratadas   2.306  1.995  1.127  1.710
      controles  2.328  2.167  1.200  1.720
250   tratadas   3.015  2.631  1.530  2.234
      controles  3.027  2.722  1.604  2.313
300   tratadas   3.710  3.209  1.881  2.763
      controles  3.683  3.319  1.920  2.802

### PIC

In [6]:
medias.query("nivel == 'PIC'").set_index(["radio", "grupo"])[PERIODOS].round(3)

antes  año 1  año 2  año 3
radio grupo                                
100   tratadas   0.947  1.004  0.875  1.192
      controles  0.889  0.908  0.759  1.003
150   tratadas   1.228  1.294  1.128  1.649
      controles  1.220  1.351  1.103  1.504
200   tratadas   1.566  1.652  1.469  2.144
      controles  1.519  1.691  1.424  2.056
250   tratadas   2.030  2.186  1.876  2.764
      controles  1.968  2.237  1.875  2.726
300   tratadas   2.480  2.677  2.284  3.367
      controles  2.300  2.654  2.201  3.119

### FCS

In [7]:
medias.query("nivel == 'FCS'").set_index(["radio", "grupo"])[PERIODOS].round(3)

antes  año 1  año 2  año 3
radio grupo                                
100   tratadas   0.015  0.015  0.013  0.011
      controles  0.014  0.016  0.011  0.013
150   tratadas   0.018  0.021  0.013  0.013
      controles  0.018  0.009  0.020  0.017
200   tratadas   0.021  0.022  0.015  0.014
      controles  0.019  0.015  0.020  0.021
250   tratadas   0.028  0.030  0.022  0.019
      controles  0.032  0.020  0.025  0.023
300   tratadas   0.031  0.035  0.028  0.021
      controles  0.036  0.025  0.026  0.033

## Efecto por año

In [8]:
def f(x, d=3):
    return "" if pd.isna(x) else f"{x:.{d}f}"


def tabla_anios(nivel, especificacion="covariables × año"):
    d = por_anio.query("nivel == @nivel and especificacion == @especificacion").set_index("radio").sort_index()
    filas = {}
    for k in (1, 2, 3):
        filas[f"Tratado × año {k}"] = [f"{f(x)}{s}" for x, s in zip(d[f"coef año {k}"], d[f"sig año {k}"])]
        filas["  (EE)" + " " * k] = [f"({f(x)})" for x in d[f"EE año {k}"]]
        filas["  % de la media del control" + " " * k] = [f(x, 2) for x in d[f"% año {k}"]]
    filas["p de igualdad entre años"] = [f(x) for x in d["p igualdad entre años"]]
    filas["Media del control, antes"] = [f(x) for x in d["media control, antes"]]
    filas["Observaciones"] = [f"{int(x):,}" for x in d.N]
    filas["R²"] = [f(x) for x in d.R2]
    filas["Clusters"] = [str(int(x)) for x in d.clusters]
    return pd.DataFrame(filas, index=[f"{r} m" for r in d.index]).T


### general

In [9]:
tabla_anios("general")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,0.074,-0.189,-0.233*,-0.163,-0.263
(EE),(0.094),(0.120),(0.141),(0.157),(0.178)
% de la media del control,3.34,-6.39,-6.04,-3.25,-4.38
Tratado × año 2,0.062,-0.076,-0.054,-0.098,-0.071
(EE),(0.123),(0.143),(0.180),(0.207),(0.252)
% de la media del control,2.81,-2.59,-1.40,-1.95,-1.17
Tratado × año 3,0.237*,0.144,0.014,-0.132,0.015
(EE),(0.137),(0.151),(0.190),(0.217),(0.240)
% de la media del control,10.66,4.87,0.37,-2.62,0.25
p de igualdad entre años,0.195,0.112,0.492,0.941,0.512


### MIN

In [10]:
tabla_anios("MIN")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,0.020,-0.142,-0.149,-0.060,-0.115
(EE),(0.071),(0.088),(0.108),(0.118),(0.133)
% de la media del control,1.53,-8.28,-6.41,-1.97,-3.13
Tratado × año 2,-0.011,-0.091,-0.056,-0.038,-0.000
(EE),(0.098),(0.110),(0.144),(0.159),(0.194)
% de la media del control,-0.82,-5.31,-2.41,-1.27,-0.01
Tratado × año 3,0.113,0.026,0.002,-0.059,-0.020
(EE),(0.097),(0.102),(0.124),(0.139),(0.163)
% de la media del control,8.58,1.52,0.09,-1.96,-0.54
p de igualdad entre años,0.083,0.237,0.552,0.977,0.787


### PIC

In [11]:
tabla_anios("PIC")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,0.055,-0.059,-0.088,-0.120,-0.162*
(EE),(0.054),(0.067),(0.076),(0.086),(0.096)
% de la media del control,6.22,-4.80,-5.80,-6.10,-7.03
Tratado × año 2,0.072,0.020,0.008,-0.060,-0.076
(EE),(0.049),(0.064),(0.072),(0.087),(0.094)
% de la media del control,8.05,1.67,0.56,-3.06,-3.32
Tratado × año 3,0.127*,0.121,0.019,-0.074,0.042
(EE),(0.066),(0.086),(0.109),(0.126),(0.132)
% de la media del control,14.32,9.95,1.27,-3.74,1.82
p de igualdad entre años,0.572,0.103,0.431,0.819,0.315


### FCS

In [12]:
tabla_anios("FCS")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,-0.001,0.012*,0.004,0.016**,0.013
(EE),(0.006),(0.006),(0.007),(0.008),(0.009)
% de la media del control,-8.76,63.39,21.86,50.56,37.53
Tratado × año 2,0.002,-0.006,-0.007,0.001,0.006
(EE),(0.005),(0.006),(0.007),(0.008),(0.007)
% de la media del control,11.08,-32.80,-35.78,2.10,17.47
Tratado × año 3,-0.004,-0.004,-0.007,0.001,-0.007
(EE),(0.005),(0.006),(0.007),(0.008),(0.009)
% de la media del control,-25.44,-20.33,-38.29,3.16,-20.15
p de igualdad entre años,0.666,0.022,0.279,0.136,0.133


## Sin covariables

### general

In [13]:
tabla_anios("general", "sin covariables")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,0.047,-0.212*,-0.232,-0.177,-0.280
(EE),(0.100),(0.125),(0.145),(0.168),(0.183)
% de la media del control,2.11,-7.18,-6.00,-3.52,-4.66
Tratado × año 2,0.019,-0.114,-0.060,-0.120,-0.157
(EE),(0.143),(0.161),(0.193),(0.230),(0.275)
% de la media del control,0.86,-3.87,-1.55,-2.39,-2.61
Tratado × año 3,0.217,0.134,0.043,-0.091,-0.005
(EE),(0.150),(0.166),(0.202),(0.244),(0.269)
% de la media del control,9.77,4.55,1.12,-1.81,-0.09
p de igualdad entre años,0.158,0.115,0.462,0.934,0.548


### MIN

In [14]:
tabla_anios("MIN", "sin covariables")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,0.011,-0.158*,-0.151,-0.079,-0.137
(EE),(0.073),(0.089),(0.107),(0.124),(0.131)
% de la media del control,0.82,-9.26,-6.47,-2.61,-3.72
Tratado × año 2,-0.040,-0.125,-0.051,-0.061,-0.066
(EE),(0.117),(0.127),(0.154),(0.179),(0.217)
% de la media del control,-3.06,-7.32,-2.19,-2.02,-1.80
Tratado × año 3,0.090,0.002,0.011,-0.067,-0.065
(EE),(0.112),(0.116),(0.134),(0.157),(0.180)
% de la media del control,6.79,0.12,0.46,-2.20,-1.78
p de igualdad entre años,0.081,0.254,0.540,0.993,0.907


### PIC

In [15]:
tabla_anios("PIC", "sin covariables")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,0.038,-0.065,-0.085,-0.112,-0.157
(EE),(0.058),(0.071),(0.079),(0.089),(0.100)
% de la media del control,4.26,-5.34,-5.58,-5.71,-6.84
Tratado × año 2,0.059,0.017,-0.001,-0.060,-0.096
(EE),(0.052),(0.067),(0.076),(0.091),(0.099)
% de la media del control,6.58,1.42,-0.10,-3.07,-4.19
Tratado × año 3,0.131*,0.136,0.042,-0.024,0.068
(EE),(0.073),(0.096),(0.122),(0.145),(0.157)
% de la media del control,14.74,11.18,2.73,-1.20,2.97
p de igualdad entre años,0.452,0.088,0.457,0.757,0.304


### FCS

In [16]:
tabla_anios("FCS", "sin covariables")

,100 m,150 m,200 m,250 m,300 m
Tratado × año 1,-0.002,0.012*,0.004,0.015*,0.014
(EE),(0.006),(0.006),(0.007),(0.008),(0.009)
% de la media del control,-12.77,63.93,19.35,45.37,39.50
Tratado × año 2,0.001,-0.006,-0.007,0.001,0.006
(EE),(0.006),(0.006),(0.007),(0.008),(0.007)
% de la media del control,6.38,-34.43,-38.71,3.70,16.81
Tratado × año 3,-0.004,-0.004,-0.009,-0.001,-0.008
(EE),(0.006),(0.006),(0.007),(0.008),(0.009)
% de la media del control,-25.53,-24.59,-48.39,-1.85,-23.53
p de igualdad entre años,0.752,0.021,0.246,0.205,0.099
